# NIG-TimeNet: the evaluation battery - can this model be trusted?

**Runtime > Run all**, or change the settings cell and re-run any single test cell. This notebook
judges a FINISHED run - it trains nothing. It auto-detects the latest run your `ar_stream` /
`panel_ar_stream` notebooks saved to Drive (`RUN_NAME` picks an explicit one) and interrogates it
from ten angles, each as its own re-runnable cell, the way a trader would before trusting money to it:

1. **Sanity** - does the predicted magnitude rank the true moves (IC), and is the direction unbiased?
2. **Confidence calibration** - is the model right more often exactly where it is confident?
3. **Anti-cowardice** - does it predict real magnitudes, or hide near zero?
4. **Trading grid** - trade the predicted direction, top-% by magnitude, with cost.
5. **Realistic execution** - limit orders AT the predicted price, market fallback.
6. **Uncertainty filter** - do low-sigma (self-confident) predictions do better?
7. **Shock regimes** - does the edge survive the wildest quintile of moves?
8. **Weekly stability** - is the edge there week after week, or one lucky month?
9. **Per-coin** - which coins carry the edge and which drown it?
10. **Flip test** - is the direction inverted?
11. **Table integrity** - missing values, sign mismatches, VAL/TEST time overlap.
12. **Persistence baseline** - does the model beat trading the last candle alone?
13. **Selective evaluation** - VAL-chosen confidence thresholds, judged on TEST.
14. **Market-neutral portfolio** - long the top coins vs shorting the bottom, per period.

Then the verdict cell weighs the best realistic strategy (TEST Sharpe) and names what to check before
believing it.


In [ ]:
TORCH_REPO, TORCH_REF = "yuosef772424/nig-timenet-torch", "feat/ar-stream"
EXPECT_DEVICE = "cuda"                         # "cpu" only for a deliberate CPU run; a cuda run on a CPU VM fails
TOKEN_PATH = "/content/drive/MyDrive/crypto/secrets/github_token.txt"
WORKDIR = "/content/nig_runner"                # clones' scratch, logs, git credentials helper: never /tmp (rule E2)
TORCH_DIR = "/content/nig-timenet-torch"
DRIVE_RUNS = "/content/drive/MyDrive/nig_timenet_runs"
EXTRA_MODULES = ()                             # torch/pandas/numpy ship with Colab
RUN_NAME = "latest"                           # "latest" (newest on Drive) or an explicit run folder name
COST = 0.0008                                  # round-trip cost per trade, as a fraction of notional
PCT = 0.10                                     # top fraction by |mu| for the single-strategy tests
TOP_PCTS = (0.05, 0.10, 0.20)                  # top-% grid for the trading-grid and limit-order sections
CONFIDENCE_BUCKETS = 5                         # confidence-calibration buckets
UNCERTAINTY_QUANTILES = (0.3, 0.5, 0.7)        # VAL sigma quantiles for the uncertainty filter
SHOCK_GROUPS = 5                               # |y| groups for the calm-to-wild shock split
MIN_WEEK_TRADES = 50                           # a week needs this many trades to be judged
SELECTIVE_QUANTILES = (0.25, 0.50, 0.75)          # VAL confidence thresholds for the selective slice
NEUTRAL_QUANTILES = (0.2, 0.3, 0.4)              # long/short tail sizes for the market-neutral portfolio
MIN_COIN_TRADES = 30                           # a coin needs this many trades to be judged


## 1. Setup
The first cell is plain code: nothing of the package exists on the VM yet, so it downloads the one stdlib-only file
`colab/bootstrap.py` with the token and runs it. Everything after it is a normal import.


In [ ]:
import getpass, os, urllib.error, urllib.request
from google.colab import drive
drive.mount("/content/drive")
token = open(TOKEN_PATH).read().strip() if os.path.exists(TOKEN_PATH) else getpass.getpass("GitHub token (not shown): ")
url = f"https://api.github.com/repos/{TORCH_REPO}/contents/src/nig_timenet_torch/colab/bootstrap.py?ref={TORCH_REF}"
req = urllib.request.Request(url, headers={"Authorization": "Bearer " + token.strip(), "User-Agent": "nig-colab",
                                           "Accept": "application/vnd.github.raw"})
try:
    boot_source = urllib.request.urlopen(req, timeout=60).read().decode()
except urllib.error.HTTPError as e:
    raise SystemExit(f"GitHub answered HTTP {e.code}: the token is missing, expired or cannot read {TORCH_REPO}. "
                     f"Delete {TOKEN_PATH} and run this cell again.")
boot = {}
exec(compile(boot_source, "bootstrap.py", "exec"), boot)
token = boot["acquire_token"](TOKEN_PATH, TORCH_REPO, lambda: getpass.getpass("GitHub token (not shown): "))


In [ ]:
shas = boot["setup_clones"](token, WORKDIR, {"torch": (TORCH_REPO, TORCH_REF, TORCH_DIR)})
install = boot["install_package"](TORCH_DIR, modules=EXTRA_MODULES)


In [ ]:
from nig_timenet_torch.colab.session import check_environment

setup = check_environment(EXPECT_DEVICE, shas, install, WORKDIR, DRIVE_RUNS)


## 2. The run: auto-detect the latest on Drive
`RUN_NAME = "latest"` picks the newest folder under `DRIVE_RUNS` that carries
`predictions/test_predictions.parquet`; an explicit name judges that run. Its VAL and TEST tables
load as-is (one row per window/coin, step).


In [ ]:
import pandas as pd
from pathlib import Path
runs = sorted(Path(DRIVE_RUNS).glob("*/predictions/test_predictions.parquet"), key=lambda p: p.stat().st_mtime)
assert runs, f"no run with predictions under {DRIVE_RUNS}: finish an ar_stream/panel_ar_stream run first"
if RUN_NAME != "latest":
    assert (Path(DRIVE_RUNS) / RUN_NAME).is_dir(), f"no run named {RUN_NAME} under {DRIVE_RUNS}"
RUN = (Path(DRIVE_RUNS) / RUN_NAME) if RUN_NAME != "latest" else runs[-1].parent.parent
V = pd.read_parquet(RUN / "predictions/val_predictions.parquet")
T = pd.read_parquet(RUN / "predictions/test_predictions.parquet")
coins = sorted(T["coin"].unique()) if "coin" in T else "single-coin run"
print("run:", RUN.name)
print(f"VAL n={len(V):,}   TEST n={len(T):,}", "coins:", coins)


## 3. The tests - each one is its own cell
عدّل خلية الإعدادات أعلاه (الكلفة، النسب، المجموعات...) ثم أعد تشغيل أي اختبار وحده - لا حاجة لإعادة
التشغيل كله. Each test below is a separate cell: change the settings cell and re-run only the test you
care about; the tables always print with a one-line reading.


### 1. Sanity - IC, direction accuracy, up-bias
هل يرتّب مقدار التنبؤ الحركات فعلاً، وهل الاتجاه متحيز للصعود؟


In [ ]:
from nig_timenet_torch.eval.battery import sanity, show

show(sanity(V, T))


### 2. Confidence calibration
هل يكون النموذج مصيباً أكثر بالضبط حيث هو واثق؟


In [ ]:
from nig_timenet_torch.eval.battery import confidence_buckets, show

show(confidence_buckets(T, n_buckets=CONFIDENCE_BUCKETS))


### 3. Anti-cowardice - does it predict real magnitudes?
هل يتنبأ بمقادير حقيقية أم يختبئ قرب الصفر؟ (ما بُنيت دالة czar لأجله)


In [ ]:
from nig_timenet_torch.eval.battery import cowardice, show

show(cowardice(T))


### 4. Trading grid - trade the direction, top-% by |mu|
تداول الاتجاه المتنبأ به مع الكلفة، أفضل النسب حسب قوة الإشارة.


In [ ]:
from nig_timenet_torch.eval.battery import show, trade_grid

show(trade_grid(T, COST, pcts=TOP_PCTS))


### 5. Realistic execution - limit order at mu, market fallback
تنفيذ واقعي: أمر حدّي عند السعر المتنبأ به، وتراجع للسوق إن لم يُلمس.


In [ ]:
from nig_timenet_torch.eval.battery import limit_execution, show

show(limit_execution(T, COST, pcts=TOP_PCTS))


### 6. Uncertainty filter - low sigma only
هل التنبؤات منخفضة عدم اليقين أفضل؟ (عتبات من VAL مطبقة على TEST دون تسريب)


In [ ]:
from nig_timenet_torch.eval.battery import show, uncertainty_filter

show(uncertainty_filter(V, T, COST, pct=PCT, quantiles=UNCERTAINTY_QUANTILES))


### 7. Shock regimes - calm to wildest
أين تعيش الميزة بالضبط: من الهدوء إلى أعلى خُمس حركة؟ (السؤال الجوهري عن الحركات غير المتوقعة)


In [ ]:
from nig_timenet_torch.eval.battery import shock_split, show

show(shock_split(T, COST, n_groups=SHOCK_GROUPS, pct=PCT))


### 8. Weekly stability
هل الميزة موجودة أسبوعاً بعد أسبوع أم شهر واحد محظوظ؟


In [ ]:
from nig_timenet_torch.eval.battery import show, weekly_stability

show(weekly_stability(T, COST, pct=PCT, min_n=MIN_WEEK_TRADES))


### 9. Per-coin
أي عملات تحمل الميزة وأيها تغرقها؟


In [ ]:
from nig_timenet_torch.eval.battery import per_coin, show

show(per_coin(T, COST, pct=PCT, min_n=MIN_COIN_TRADES))


### 10. Flip test
هل الاتجاه مقلوب؟ قارن التداول مع الإشارة ضد التداول بها معكوسة.


In [ ]:
from nig_timenet_torch.eval.battery import flip_test, show

show(flip_test(T, COST, pct=PCT))


### 11. Table integrity - the data first
سلامة الجدول نفسه: لا قيم مفقودة، لا تعارض إشارة، ولا تداخل زمني بين VAL وTEST.


In [ ]:
from nig_timenet_torch.eval.battery import show, table_integrity

show(table_integrity(V, T))


### 12. Persistence baseline - the last candle alone
هل يضيف النموذج شيئاً فوق الشمعة الأخيرة؟ خط أساس الزخم هو أرخص حافة ممكنة.


In [ ]:
from nig_timenet_torch.eval.battery import persistence_baseline, show

show(persistence_baseline(T, COST, pct=PCT))


### 13. Selective evaluation - VAL thresholds, TEST judgment
العتبات تُختار على VAL فقط ثم تُقاس على TEST: هل ترتفع الدقة مع الثقة فعلاً أم أن VAL يجامل نفسه؟


In [ ]:
from nig_timenet_torch.eval.battery import selective_evaluation, show

show(selective_evaluation(V, T, COST, quantiles=SELECTIVE_QUANTILES))


### 14. Market-neutral portfolio - long top, short bottom
محفظة محايدة للسوق: ترتيب العملات كل فترة حسب الإشارة، شراء الأعلى وبيع الأدنى (تحتاج تشغيل لوحة بأربع عملات على الأقل).


In [ ]:
from nig_timenet_torch.eval.battery import market_neutral, show

show(market_neutral(V, T, COST, quantiles=NEUTRAL_QUANTILES))


## 4. The verdict
الحكم النهائي: أفضل استراتيجية واقعية (Sharpe على TEST) مع تنبيهات أقسام الصدمات والاستقرار الأسبوعي.


In [ ]:
from nig_timenet_torch.eval.battery import full_battery, show

out = full_battery(V, T, cost=COST)
show(out["verdict"])


## 5. How to read it
* An IC near zero in section 1 means the magnitude carries no ranking information - only the direction can pay.
* Section 3's ratio well under 1 is the coward signature the czar loss is built against; well over 1 means the
  model overshoots and the limit orders in section 5 will rarely fill.
* Section 7 answers the owner's core question: the accuracy is EXPECTED to decay in the wildest quintile
  (news, thin liquidity, squeezes are not pattern); what matters is that the PnL does not explode there.
* The verdict's thresholds (Sharpe 0.10 / 0.05 per trade) are per-trade, not annualized: read them as a
  ranking across strategies, and demand the weekly section agree before calling anything an edge.
